# KonfAI IMPACT-Reg Demo — patch-tiled, overlap-blended registration

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fideus-labs/KonfAI/blob/main/examples/ImpactReg/register_demo.ipynb)

This notebook is meant to work from a **fresh environment**, including **Google Colab**.

[IMPACT-Reg](https://github.com/vboussot/ImpactLoss) runs pairwise image registration through the KonfAI
runtime: each *preset* (FireANTs, ConvexAdam, elastix, …) is a self-contained KonfAI app that produces,
on the **fixed** grid, the moving image resampled onto the fixed image (`Moved`) and the displacement
field (`DVF`).

Here we:

- build a **synthetic** fixed/moving pair with a **known** smooth deformation (no download needed),
- inspect the initial misalignment (overlay, checkerboard, NCC),
- register with the **FireANTs SyN** preset under **patch tiling + overlap blending** — the way a volume
  too large to register in one shot is handled: KonfAI tiles it into overlapping patches, registers each,
  and reassembles the moved image / displacement field with a `Cosinus` partition-of-unity window,
- verify quantitatively that the warp is recovered (NCC ↑) and that the blend leaves **no patch seam**.

The known deformation is a low-frequency field of amplitude ~4 voxels, so a correct registration must
drive `NCC(moved, fixed)` well above the `NCC(moving, fixed)` baseline.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules


def find_repo_root(start: Path):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "examples").exists():
            return candidate
    return None


if IN_COLAB:
    REPO_DIR = Path("/content/KonfAI")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "https://github.com/fideus-labs/KonfAI", str(REPO_DIR)], check=True)
else:
    REPO_DIR = find_repo_root(Path.cwd().resolve())
    if REPO_DIR is None:
        raise RuntimeError(
            "Could not locate the KonfAI repository. Run this notebook from the repo or open it in Google Colab."
        )

# Install the IMPACT-Reg CLI (and its plotting/imaging deps) if they are missing.
for module, package in [("impact_reg_konfai", str(REPO_DIR / "apps" / "impact_reg")), ("matplotlib", "matplotlib")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

EXAMPLE_DIR = REPO_DIR / "examples" / "ImpactReg"
DATASET_DIR = EXAMPLE_DIR / "Dataset"
DATASET_DIR.mkdir(parents=True, exist_ok=True)
print("Repository:", REPO_DIR)
print("Example dir:", EXAMPLE_DIR)

## 1. Build a demo fixed/moving pair with a known deformation

`moving` is `fixed` warped by a smooth, low-frequency displacement field (amplitude ~4 voxels). Because we
know the warp, we can measure exactly how much of it the registration recovers. The volume is `96³` — larger
than the `64³` patch used below, so the registration must tile and blend.

In [ ]:
import numpy as np
import SimpleITK as sitk
from scipy.ndimage import map_coordinates

SIDE, AMPLITUDE = 96, 4.0
shape = (SIDE, SIDE, SIDE)


def sphere(center, radius, value):
    zz, yy, xx = np.ogrid[:SIDE, :SIDE, :SIDE]
    m = (zz - center[0]) ** 2 + (yy - center[1]) ** 2 + (xx - center[2]) ** 2 <= radius**2
    return m.astype(np.float32) * value


z, y, x = np.mgrid[0:SIDE, 0:SIDE, 0:SIDE].astype(np.float32)
# rich, non-symmetric content so registration has signal everywhere, not just one blob
fixed = 40.0 + 25.0 * np.sin(x / 9.0) * np.cos(y / 11.0) + 0.15 * z
for c, r, v in [((32, 32, 32), 12, 220.0), ((64, 64, 64), 16, 300.0), ((72, 24, 64), 9, 160.0)]:
    fixed = np.maximum(fixed, sphere(c, r, v))
fixed = fixed.astype(np.float32)

# known smooth displacement field -> warp fixed into moving
dz = AMPLITUDE * np.sin(x / 18.0) * np.cos(y / 20.0)
dy = AMPLITUDE * np.sin(y / 16.0) * np.cos(z / 22.0)
dx = AMPLITUDE * np.cos(x / 14.0) * np.sin(z / 19.0)
moving = map_coordinates(fixed, [z + dz, y + dy, x + dx], order=1, mode="nearest").astype(np.float32)

for name, arr in [("fixed", fixed), ("moving", moving)]:
    img = sitk.GetImageFromArray(arr)
    img.SetSpacing((1.0, 1.0, 1.0))
    sitk.WriteImage(img, str(DATASET_DIR / f"{name}.mha"))


def ncc(a, b):
    a, b = a - a.mean(), b - b.mean()
    return float((a * b).sum() / (np.sqrt((a**2).sum() * (b**2).sum()) + 1e-8))


print(f"volume {shape}  known warp |d| mean={np.linalg.norm(np.stack([dz, dy, dx]), axis=0).mean():.2f} vox")
print(f"baseline NCC(moving, fixed) = {ncc(moving, fixed):.4f}")

## 2. Inspect the initial misalignment

A magenta/green overlay shows where `fixed` and `moving` disagree; the checkerboard alternates 8×8 tiles of
each; the difference map is `|fixed − moving|`. All three should look misaligned before registration.

In [ ]:
import matplotlib.pyplot as plt


def overlay(ax, a, b, title):
    rgb = np.zeros((*a.shape, 3), dtype=np.float32)
    rgb[..., 0] = rgb[..., 2] = a / (a.max() + 1e-8)  # magenta = first image
    rgb[..., 1] = b / (b.max() + 1e-8)                # green = second image
    ax.imshow(np.clip(rgb, 0, 1)); ax.set_title(title); ax.axis("off")


def checkerboard(a, b, tiles=8):
    step = a.shape[0] // tiles
    board = a.copy()
    for i in range(tiles):
        for j in range(tiles):
            if (i + j) % 2:
                board[i * step:(i + 1) * step, j * step:(j + 1) * step] = b[i * step:(i + 1) * step, j * step:(j + 1) * step]
    return board


mid = SIDE // 2
fx, mv = fixed[mid], moving[mid]
fig, axes = plt.subplots(1, 3, figsize=(13, 4.4))
overlay(axes[0], fx, mv, "overlay  (fixed=magenta, moving=green)")
axes[1].imshow(checkerboard(fx, mv), cmap="gray"); axes[1].set_title("checkerboard  fixed / moving"); axes[1].axis("off")
axes[2].imshow(np.abs(fx - mv), cmap="magma"); axes[2].set_title(f"|fixed - moving|   NCC={ncc(moving, fixed):.3f}"); axes[2].axis("off")
plt.tight_layout(); plt.show()

## 3. Register with IMPACT-Reg (FireANTs SyN), patch-tiled + blended

The `--set` overrides force patch-based inference: `Patch.patch_size=[64,64,64]`, `Patch.overlap=16`, and a
`Cosinus` partition-of-unity blend on **both** outputs (moved image and displacement field). The FireANTs
iteration counts are trimmed to keep the 8-patch demo quick. Every override is forwarded verbatim to
`konfai-apps infer --set`, so the exact same command scales to any preset or any real volume.

> The presets are external model apps on the Hugging Face repo `VBoussot/ImpactReg` and FireANTs needs a
> CUDA GPU. Set `KONFAI_IMPACTREG_REPO` to a local bundle directory to run offline.

In [ ]:
import os
import shutil
import torch

PRESET = "FireANTs_SyN"
OUTPUT_DIR = EXAMPLE_DIR / "Output"
device_args = ["--gpu", "0"] if torch.cuda.is_available() else ["--cpu", "1"]

REGISTER_CMD = [
    "impact-reg-konfai", "register",
    PRESET,  # one or more presets, positional; several presets are ensembled
    "-f", str(DATASET_DIR / "fixed.mha"),
    "-m", str(DATASET_DIR / "moving.mha"),
    "-o", str(OUTPUT_DIR),
    *device_args,
    "--set", "Predictor.Dataset.Patch.patch_size=[64, 64, 64]",
    "--set", "Predictor.Dataset.Patch.overlap=16",
    "--set", "Predictor.outputs_dataset.MovedImage.OutputDataset.patch_combine=Cosinus",
    "--set", "Predictor.outputs_dataset.DisplacementField.OutputDataset.patch_combine=Cosinus",
    "--set", "affine_iterations=[100, 50, 25]",
    "--set", "deformable_iterations=[100, 50, 25]",
]
print("$", " ".join(REGISTER_CMD))

In [ ]:
RUN_REGISTER = True  # set False to only review the command above

if RUN_REGISTER and PRESET.startswith("FireANTs") and not torch.cuda.is_available():
    print(
        f"Skipping registration: the '{PRESET}' preset (FireANTs) needs a CUDA GPU and none is available.",
        "\n- In Google Colab: Runtime > Change runtime type > Hardware accelerator = GPU, then re-run.",
        "\n- Or switch PRESET above to a CPU-capable preset (e.g. ConvexAdam_Fine or Generic_Rigid).",
    )
elif RUN_REGISTER:
    try:
        subprocess.run(REGISTER_CMD, check=True)
        print("\nRegistration done ->", OUTPUT_DIR / "P000")
    except (FileNotFoundError, subprocess.CalledProcessError) as error:
        print(
            "Registration did not run:", error,
            "\nNeed the IMPACT-Reg CLI + a FireANTs GPU preset. Install `apps/impact_reg`, ensure a CUDA GPU,",
            "and either keep network access to `VBoussot/ImpactReg` or set KONFAI_IMPACTREG_REPO to a local bundle.",
        )

## 4. Inspect the result — moved image, field, and a seam-free blend

`NCC(moved, fixed)` should be well above the baseline, the checkerboard should look continuous, and the
recovered displacement field (quiver) should mirror the known warp. The seam check compares the field's
gradient on the patch-boundary planes against the global mean: a hard seam would spike, a partition-of-unity
blend does not.

In [ ]:
moved_path = OUTPUT_DIR / "P000" / "Moved.mha"
dvf_path = OUTPUT_DIR / "P000" / "DVF.mha"

if moved_path.is_file() and dvf_path.is_file():
    moved = sitk.GetArrayFromImage(sitk.ReadImage(str(moved_path))).astype(np.float32)
    dvf = sitk.GetArrayFromImage(sitk.ReadImage(str(dvf_path))).astype(np.float32)  # (Z, Y, X, 3)
    print(f"NCC(moving, fixed) = {ncc(moving, fixed):.4f}   ->   NCC(moved, fixed) = {ncc(moved, fixed):.4f}")

    fig, axes = plt.subplots(1, 3, figsize=(13, 4.4))
    axes[0].imshow(checkerboard(fixed[mid], moved[mid]), cmap="gray")
    axes[0].set_title(f"checkerboard after   NCC={ncc(moved, fixed):.3f}"); axes[0].axis("off")

    step = 5
    yy, xx = np.mgrid[0:SIDE:step, 0:SIDE:step]
    u = dvf[mid, ::step, ::step, 2]  # x-component
    v = dvf[mid, ::step, ::step, 1]  # y-component
    axes[1].imshow(fixed[mid], cmap="gray")
    axes[1].quiver(xx, yy, u, v, np.hypot(u, v), cmap="viridis", scale=60)
    axes[1].set_title("recovered displacement field"); axes[1].axis("off")

    grad = np.abs(np.diff(dvf, axis=0)).sum(-1)
    global_mean = float(grad.mean())
    seams = [p for p in (31, 32, 63, 64) if p < grad.shape[0]]
    seam_profile = grad.reshape(grad.shape[0], -1).mean(axis=1)
    axes[2].plot(seam_profile, color="steelblue")
    for p in seams:
        axes[2].axvline(p, color="crimson", ls="--", lw=0.8)
    axes[2].axhline(global_mean, color="gray", ls=":", label="global mean")
    axes[2].set_title("DVF z-gradient (red = patch borders)"); axes[2].set_xlabel("z"); axes[2].legend()
    plt.tight_layout(); plt.show()

    seam_max = max(float(grad[p].mean()) for p in seams)
    print(f"seam check: worst patch-border gradient {seam_max:.3f} vs global mean {global_mean:.3f} "
          f"({seam_max / (global_mean + 1e-8):.2f}x)  ->  {'no seam' if seam_max < 6 * global_mean else 'SEAM!'}")
else:
    print("Run the registration cell above first (RUN_REGISTER = True).")

## 5. Expected outputs & next steps

After `impact-reg-konfai register` you get, under `Output/P000/`:

- `Moved.mha` — the moving image resampled onto the fixed grid,
- `DVF.mha` — the displacement field on the fixed grid,
- `Transform.h5` — the same field as a SimpleITK transform (consumed by `evaluate` and SlicerImpactReg).

On this demo the patch-tiled + Cosinus-blended SyN preset recovers the known warp (`NCC ≈ 0.84 → ≈ 0.99`)
with no patch seam.

From here you can:

- **swap the preset** — `register ConvexAdam_Fine ...` (itk-impact) or `register Generic_Rigid ...` (elastix);
- **ensemble** several presets — pass them as positionals, `register FireANTs_SyN ConvexAdam_Fine ...`; the
  displacement fields are averaged and, with `--uncertainty`, each preset's field is kept (under `Ensemble/`)
  for an `impact-reg-konfai uncertainty` spread map;
- **evaluate** against ground truth — `impact-reg-konfai evaluate` scores image MAE, segmentation Dice, or
  landmark TRE;
- **register real volumes** — the same command works on OME-Zarr or DICOM inputs (KonfAI auto-detects the
  store format on read), and the patch/overlap you set here is exactly how a volume too large for one pass
  is handled.